

```
# This is formatted as code
```

# Building an RAG Pipeline Using PDF Chunking and Retrieval:

# Step 1: Install required packages

In [ ]:
!pip install -U langchain langchain-google-genai langchain-community langchain-classic faiss-cpu tiktoken pypdf

# Step 2: Import dependencies

In [ ]:
import os
import glob
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA

# Step 3: Set Google Gemini credentials



In [ ]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Step 4: Load and chunk your custom FAQ document


In [ ]:
# Load and chunk the three PDF documents

pdf_paths = ["pdf1.pdf", "pdf2.pdf", "pdf3.pdf"]

documents = []
for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents.extend(loader.load())

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_splitter.split_documents(documents)

print(f"Loaded {len(pdf_paths)} PDFs into {len(docs)} chunks")

# Step 5: Create vectorstore using Google embeddings


In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    task_type="retrieval_document",
)

vectorstore = FAISS.from_documents(docs, embeddings)

# Step 6: Initialize the Google Gemini LLM

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    temperature=0,
)

# Step 7: Create the RAG chain

In [ ]:
# Create a RetrievalQA chain that uses the retriever and LLM to answer queries with source context
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectorstore.as_retriever(),
    return_source_documents=True
)

# Step 8: Ask a question

In [ ]:
# Send a question to the RAG chain and store the result
query = "How do you foster a customer-centric culture?"  # replace with a question relevant to your PDFs
result = qa_chain.invoke({"query": query})

# Step 9: Print results

In [ ]:
# Display the final answer and the source chunks used

print("Answer:", result["result"])

print("\n--- Sources ---")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\nSource {i} (page {doc.metadata.get('page', 'n/a')}, from {os.path.basename(doc.metadata.get('source', ''))}):")
    print(doc.page_content)

# Chunking metrics required by the project report
total_chunks = len(docs)
avg_chunk_size = sum(len(d.page_content) for d in docs) / total_chunks
print(f"\n--- Metrics ---")
print(f"Total chunks: {total_chunks}")
print(f"Average chunk size: {avg_chunk_size:.1f} characters")